# Lettuce Growth Stage Object Detection: YOLOv8 Pipeline
This notebook retrieves bounding-box annotated data, performs Exploratory Data Analysis, trains a YOLOv8 Nano model, evaluates test metrics, and exports the model to NCNN format for Android deployment.

In [2]:
# Install the necessary libraries for YOLO and Roboflow
%pip install ultralytics roboflow pyyaml -q

import os
import yaml
from ultralytics import YOLO
from roboflow import Roboflow


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import yaml
from ultralytics import YOLO
from roboflow import Roboflow

In [ ]:
# Fetch Dataset in YOLOv8 Format (https://docs.roboflow.com/datasets/create-and-upload/download-a-dataset)
# Replace 'YOUR_API_KEY' with your actual Roboflow API key
rf = Roboflow(api_key="Qb9gtEx8C69ECxGhT4js")
project = rf.workspace("robotics-v9fnj").project("lettuce-growth-stage-qld0b")

#https://github.com/roboflow/notebooks/blob/main/notebooks/train-yolov8-object-detection-on-custom-dataset.ipynb

dataset = project.version(1).download("yolov8")

data_yaml_path = os.path.join(dataset.location, "data.yaml")

#https://developer.ibm.com/tutorials/awb-train-yolo-object-detection-model-in-python/
# Load class names for later evaluation
with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg.get("names", [])
if isinstance(class_names, dict):
    class_names = list(class_names.values())

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
from collections import Counter

def run_yolo_eda(dataset_loc, classes):
    print("EXPLORATORY DATA ANALYSIS: LETTUCE GROWTH STAGES")   
    
    split_dir = os.path.join(dataset_loc, "train")
    lbl_dir = os.path.join(split_dir, "labels")
    
    if not os.path.exists(lbl_dir):
        print("Labels directory not found. Ensure dataset downloaded correctly.")
        return

    class_counts = Counter()
    total_boxes = 0

    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith(".txt"):
            continue
            
        with open(os.path.join(lbl_dir, lbl_file), "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 0:
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1
                    total_boxes += 1

    print(f"Total Bounding Boxes in Training Set: {total_boxes}")
    print("-" * 65)
    print(f"{'Class ID':<10} {'Stage Name':<18} {'Box Count':<12} {'Percentage'}")
    print("-" * 65)
    
    for cls_id in range(len(classes)):
        count = class_counts[cls_id]
        pct = (count / total_boxes * 100) if total_boxes > 0 else 0
        print(f"{cls_id:<10} {classes[cls_id]:<18} {count:<12} {pct:.1f}%")

run_yolo_eda(dataset.location, class_names)

      EXPLORATORY DATA ANALYSIS: LETTUCE GROWTH STAGES       
Total Bounding Boxes in Training Set: 623
-----------------------------------------------------------------
Class ID   Stage Name         Box Count    Percentage
-----------------------------------------------------------------
0          Harvest Stage      218          35.0%
1          Seedling Stage     209          33.5%
2          Vegetative Stage   196          31.5%


In [6]:
print("Starting YOLOv8 Training")

model = YOLO("yolov8n.pt")
#https://learnopencv.com/train-yolov8-on-custom-dataset/
#https://medium.com/data-science/the-comprehensive-guide-to-training-and-running-yolov8-models-on-custom-datasets-22946da259c3
#https://roboflow.com/blog/how-to-train-yolov8-on-a-custom-dataset
results = model.train(
    data=data_yaml_path,
    epochs=50,             
    imgsz=320,             
    batch=16,              
    device="cpu",          
    name="lettuce_live_feed",
    patience=10,           # Equivalent to Keras EarlyStopping
    plots=True             # Automatically generates accuracy/loss plots
)

--- Starting YOLOv8 Training ---
New https://pypi.org/project/ultralytics/8.4.185 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.173  Python-3.13.16 torch-2.14.1+cpu CPU (13th Gen Intel Core i7-1355U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Research_RPDA_8412w\Lettuce-Growth-Stage-1\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=

In [7]:
print("\nEvaluating Model on Test Set")

# Run validation on the test split
metrics = model.val(
    data=data_yaml_path,
    split="test",
    imgsz=320
)
#https://roboflow.com/blog/mean-average-precision
#https://jonathan-hui.medium.com/map-mean-average-precision-for-object-detection-45c121a31173
# https://www.geeksforgeeks.org/computer-vision/mean-average-precision-map-in-computer-vision/
# https://www.kaggle.com/discussions/getting-started/570104

print("MODEL PERFORMANCE METRICS (TEST SET)")
print()
print(f"Overall mAP@50 (Mean Average Precision): {metrics.box.map50:.4f}")
print()
print(f"{'Growth Stage':<18} {'Precision':<12} {'Recall':<12} {'mAP@50':<12}")
print()

for i, stage_name in enumerate(class_names):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    print(f"{stage_name:<18} {p:<12.4f} {r:<12.4f} {map50:<12.4f}")


Evaluating Model on Test Set
Ultralytics 8.4.173  Python-3.13.16 torch-2.14.1+cpu CPU (13th Gen Intel Core i7-1355U)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.1 ms, read: 15.95.4 MB/s, size: 19.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Research_RPDA_8412w\Lettuce-Growth-Stage-1\test\labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 7.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.5it/s 0.8s1.2s
                   all         30         30       0.99          1      0.995      0.938
         Harvest Stage          8          8      0.985          1      0.995      0.995
        Seedling Stage          9          9      0.995          1      0.995      0.831
      Vegetative Sta

In [8]:
print(metrics.box.map, metrics.box.mp, metrics.box.mr) 
print(metrics.speed)
model.info()

0.9379490740740742 0.990256505415656 1.0
{'preprocess': 0.37915666665260994, 'inference': 14.892773332151894, 'loss': 8.666732658942541e-05, 'postprocess': 0.5131399996268252}
Model summary: 129 layers, 3,011,433 parameters, 0 gradients, 8.2 GFLOPs


(129, 3011433, 0, 8.1938432)

In [9]:
print("--- Exporting Model for Android Deployment ---")

# Export the trained model to NCNN format
# This creates a highly optimized format ready to be loaded into your Android project
export_path = model.export(format="ncnn", imgsz=320)

print(f"Model successfully exported for Android to: {export_path}")

--- Exporting Model for Android Deployment ---
Ultralytics 8.4.173  Python-3.13.16 torch-2.14.1+cpu CPU (13th Gen Intel Core i7-1355U)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 2.0 GFLOPs

PyTorch: starting from 'C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.9 MB)

NCNN: starting export with NCNN 1.0.20260526 and PNNX 20260526...


`torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
`torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.


NCNN: export success  6.9s, saved as 'C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best_ncnn_model' (11.6 MB)

Export complete (7.2s)
Results saved to C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best_ncnn_model
Predict:         yolo predict task=detect model=C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best_ncnn_model imgsz=320 
Validate:        yolo val task=detect model=C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best_ncnn_model imgsz=320 data=c:\Research_RPDA_8412w\Lettuce-Growth-Stage-1\data.yaml  
Visualize:       https://netron.app
Model successfully exported for Android to: C:\Research_RPDA_8412w\runs\detect\lettuce_live_feed-2\weights\best_ncnn_model


In [10]:
m = model.val(data=r"c:\Research_RPDA_8412w\Lettuce-Growth-Stage-1\data.yaml",
              split="test", imgsz=320, conf=0.25, plots=True)

Ultralytics 8.4.173  Python-3.13.16 torch-2.14.1+cpu CPU (13th Gen Intel Core i7-1355U)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.50.2 ms, read: 19.84.7 MB/s, size: 19.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Research_RPDA_8412w\Lettuce-Growth-Stage-1\test\labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 9.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.3it/s 0.6s0.9s
                   all         30         30          1          1      0.995      0.938
         Harvest Stage          8          8          1          1      0.995      0.995
        Seedling Stage          9          9          1          1      0.995      0.831
      Vegetative Stage         13         13      